#### This notebook serves as a data extraction of Fermi sources (all classes) for building multi-class classification model

In [1]:
import numpy as np
from astropy.table import Table
import os

# 1. Load the 4FGL-DR4 FITS catalog
# Ensure you have downloaded 'gll_psc_v35.fit' from the Fermi Science Support Center
catalog_file = 'gll_psc_v35.fit'
cat = Table.read(catalog_file, hdu=1)

In [2]:
# 2. Extract and clean the CLASS1 column
# We lowercase the strings to merge associated ('bll') and identified ('BLL') sources
class1_raw = cat['CLASS1'].astype(str)
class1_clean = np.char.strip(np.char.lower(class1_raw))

# Replace empty strings with 'unassociated' for file naming purposes
class1_clean[class1_clean == ''] = 'unassociated'

# Append the cleaned class back to the table
cat['CLASS_CLEAN'] = class1_clean

In [3]:
# 3. Get unique classes and their total source counts
unique_classes, counts = np.unique(class1_clean, return_counts=True)

# Create an output directory for the split files
output_dir = '4fgl_classes'
os.makedirs(output_dir, exist_ok=True)

In [4]:
# 4. Loop through each unique class, filter the table, and export
print(f"Found {len(unique_classes)} distinct classes. Exporting files...")
for cls in unique_classes:
    # Filter table for the current class
    mask = (class1_clean == cls)
    sub_table = cat[mask]
    
    # Define output filenames
    base_name = f"4fgl_dr4_{cls}"
    fits_out = os.path.join(output_dir, f"{base_name}.fits")
    csv_out = os.path.join(output_dir, f"{base_name}.csv")
    
    # Save the filtered data to a FITS file (retains multi-dimensional columns)
    sub_table.write(fits_out, format='fits', overwrite=True)
    
    # --- MODIFICATION FOR CSV EXPORT ---
    # Create a copy specifically for CSV export
    csv_table = sub_table.copy()
    
    # Identify and drop multi-dimensional columns (like Flux_Band)
    nd_cols = [col for col in csv_table.colnames if len(csv_table[col].shape) > 1]
    csv_table.remove_columns(nd_cols)
    
    # Save the flattened data to a standard CSV file
    csv_table.write(csv_out, format='csv', overwrite=True)
    # -----------------------------------
    
    print(f"Saved '{cls}': {len(sub_table)} sources")

Found 25 distinct classes. Exporting files...
Saved 'agn': 8 sources
Saved 'bcu': 1623 sources
Saved 'bin': 10 sources
Saved 'bll': 1490 sources
Saved 'css': 6 sources
Saved 'fsrq': 820 sources
Saved 'gal': 6 sources
Saved 'gc': 1 sources
Saved 'glc': 41 sources
Saved 'hmb': 11 sources
Saved 'lmb': 9 sources
Saved 'msp': 179 sources
Saved 'nlsy1': 8 sources
Saved 'nov': 8 sources
Saved 'psr': 141 sources
Saved 'pwn': 22 sources
Saved 'rdg': 53 sources
Saved 'sbg': 8 sources
Saved 'sey': 4 sources
Saved 'sfr': 6 sources
Saved 'snr': 44 sources
Saved 'spp': 132 sources
Saved 'ssrq': 2 sources
Saved 'unass': 2423 sources
Saved 'unk': 140 sources


In [5]:
# 5. Generate and save the summary CSV list
summary_table = Table([unique_classes, counts], names=('Class', 'Source_Count'))
# Sort by highest source count first
summary_table.sort('Source_Count', reverse=True) 
summary_table.write('4fgl_dr4_class_summary.csv', format='csv', overwrite=True)

print("\nDone! Summary list saved to '4fgl_dr4_class_summary.csv'")


Done! Summary list saved to '4fgl_dr4_class_summary.csv'
